# CISM Tutorial 01: Data Preparation

This notebook is the first step in the CISM workflow.

Its only goal is to help you finish with a dataset folder that contains:

- one `Patient_<patient_id>_FOV<fov>.txt` file per field of view
- one `patient_class.csv`

Once you have that folder, you are ready for the next stage: FANMOD+ and `CISM` initialization.

## What This Notebook Covers

CISM supports three preparation routes:

1. **Annotated centroids table** -> build spatial graph -> write colored edge-list txt files
2. **Annotated edge table** -> write colored edge-list txt files directly
3. **Prebuilt graphs** -> convert graphs to colored edge-list txt files

Run **only the section that matches your raw data**.

At the end, use the validation section to confirm that your folder is ready for `CISM.add_dataset(...)`.

## Output Contract

The target folder should look like this:

```text
data/
  my_dataset/
    Patient_1_FOV1.txt
    Patient_1_FOV2.txt
    Patient_2_FOV1.txt
    ...
    patient_class.csv
```

Each `txt` file must contain rows in this format:

```text
<src_id> <dst_id> <src_type_id> <dst_type_id>
```

`patient_class.csv` must map each patient to its class label or value.

In [1]:
from pathlib import Path

import networkx as nx
import pandas as pd

from cism import (
    prepare_from_centroids,
    prepare_from_edge_annotations,
    prepare_from_graphs,
    validate_network_dataset_directory,
)


In [2]:
# Edit these paths once and reuse them throughout the notebook.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "tutorials" else Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / "pet_data" # Change this to your data directory if it's not in the project root.
DATASET_NAME = "TNBC"
OUTPUT_DATASET_DIR = DATA_ROOT / DATASET_NAME

OUTPUT_DATASET_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Prepared dataset output directory: {OUTPUT_DATASET_DIR}")

# These are the exact handoff values you will need in tutorial 02.
network_dataset_root_path = str(DATA_ROOT)
dataset_folder = DATASET_NAME

print(f"network_dataset_root_path = {network_dataset_root_path}")
print(f"dataset_folder = {dataset_folder}")


Project root: /home/omriavit/CISM/CISM_paper_revision
Prepared dataset output directory: /home/omriavit/CISM/CISM_paper_revision/pet_data/TNBC
network_dataset_root_path = /home/omriavit/CISM/CISM_paper_revision/pet_data
dataset_folder = TNBC


In [3]:
def print_next_step_handoff(prep_result=None):
    print("Use these values in tutorial 02:")
    print(f"network_dataset_root_path = {network_dataset_root_path}")
    print(f"dataset_folder = {dataset_folder}")
    print(f"dataset directory on disk = {OUTPUT_DATASET_DIR}")
    if prep_result is not None:
        common_cells_type = {idx: cell_type for cell_type, idx in prep_result.cell_type_to_id.items()}
        print(f"number of graph txt files written = {len(prep_result.files)}")
        print(f"preparation route = {prep_result.route}")
        print("common_cells_type =")
        print(common_cells_type)


## Shared Notes Before You Start

- `patient_id` and `fov` together define one output graph file.
- The preparation functions normalize column names on a copy, so your raw dataframe does not need to already use CISM's canonical names.
- If your cell-type labels are already exactly what you want, use `cell_type_mapper=None`.
- If you want to merge or rename cell types, pass a mapping dictionary.
- The output of every route is a folder of `txt` graph files, not a `CISM` object yet.

## Route 1: Start From Annotated Centroids

Use this section if your raw data is a table of cells with:

- patient id
- FOV id
- x/y centroid coordinates
- cell type

This route builds a spatial graph and then writes one `txt` file per FOV.

In [4]:
# Example: replace with your own centroid file.
raw_centroids = pd.read_csv("../pet_data/mibitof_tnbc_pred_cell_type.csv", index_col=0)

# The original TNBC tutorial excludes patient 38 because survival-days metadata is unavailable.
raw_centroids = raw_centroids[raw_centroids["patient_id"] != 38].copy()

# Rebuilding this route should replace stale generated graph files from earlier runs.
for stale_graph_file in OUTPUT_DATASET_DIR.glob("Patient_*_FOV*.txt"):
    stale_graph_file.unlink()

# Example column mapping from raw names to the canonical preparation schema.
# Keys are the column names in your raw data, and values are the expected column names for CISM preparation.
centroid_column_mapping = {
    "patient_id": "patient_id",
    "fov": "fov",
    "cell_type": "pred",
    "centroid-0": "centroid-0",
    "centroid-1": "centroid-1",
}

# One working TNBC mapper: raw prediction id -> biological cell-type label.
# This matches the original MIBI-TOF TNBC tutorial. The graph writer then encodes
# these names to stable color ids 0..15 and prints that dictionary below.
tnbc_pred_to_cell_type = {
    1: "Unidentified",
    2: "Endothelial",
    3: "Mesenchyme",
    4: "Tumor",
    5: "Tregs",
    6: "CD4 t cells",
    7: "CD8 T cells",
    8: "CD3 T cells",
    9: "NK cells",
    10: "B cells",
    11: "Neutrophils",
    12: "Macrophages",
    13: "DC",
    14: "DC/Mono",
    15: "Mono/Neu",
    16: "Immune other",
}

prep_result = prepare_from_centroids(
    dataframe=raw_centroids,
    path_to_output_dir=str(OUTPUT_DATASET_DIR),
    column_mapping=centroid_column_mapping,
    cell_type_mapper=tnbc_pred_to_cell_type,
    max_distance=100,
    exclude_cell_type=None,
)
prep_result
print_next_step_handoff(prep_result)


{'B cells': 0, 'CD3 T cells': 1, 'CD4 t cells': 2, 'CD8 T cells': 3, 'DC': 4, 'DC/Mono': 5, 'Endothelial': 6, 'Immune other': 7, 'Macrophages': 8, 'Mesenchyme': 9, 'Mono/Neu': 10, 'NK cells': 11, 'Neutrophils': 12, 'Tregs': 13, 'Tumor': 14, 'Unidentified': 15}
Use these values in tutorial 02:
network_dataset_root_path = /home/omriavit/CISM/CISM_paper_revision/pet_data
dataset_folder = TNBC
dataset directory on disk = /home/omriavit/CISM/CISM_paper_revision/pet_data/TNBC
number of graph txt files written = 38
preparation route = centroids
common_cells_type =
{0: 'B cells', 1: 'CD3 T cells', 2: 'CD4 t cells', 3: 'CD8 T cells', 4: 'DC', 5: 'DC/Mono', 6: 'Endothelial', 7: 'Immune other', 8: 'Macrophages', 9: 'Mesenchyme', 10: 'Mono/Neu', 11: 'NK cells', 12: 'Neutrophils', 13: 'Tregs', 14: 'Tumor', 15: 'Unidentified'}


## Route 2: Start From Annotated Edges

Use this section if your raw data already contains graph edges, meaning each row already describes a pair of connected cells.

Required logical fields are:

- `patient_id`
- `fov`
- `source_id`
- `target_id`
- `source_type`
- `target_type`

This route skips graph construction and writes the CISM/FANMOD input files directly.

In [5]:
# Example: replace with your own edge-annotation file.
# raw_edges = pd.read_csv("path/to/your_edge_annotations.csv")

edge_column_mapping = {
    "patient_id": "patient",
    "fov": "fov",
    "source_id": "id1",
    "target_id": "id2",
    "source_type": "type1",
    "target_type": "type2",
}

# Uncomment once raw_edges is loaded.
# prep_result = prepare_from_edge_annotations(
#     dataframe=raw_edges,
#     path_to_output_dir=str(OUTPUT_DATASET_DIR),
#     column_mapping=edge_column_mapping,
# )
# prep_result
# print_next_step_handoff(prep_result)


## Route 3: Start From Prebuilt Graphs

Use this section if you already have one graph per patient/FOV.

Each graph must be a `networkx.Graph`, and each node must have a node attribute that stores the cell type.

By default, the preparation code expects that attribute to be called `cell_type`.

In [6]:
# Example in-memory graph.
# Replace this with your actual graph loading logic.
example_graph = nx.Graph()
example_graph.add_node("A", cell_type="Tumor")
example_graph.add_node("B", cell_type="TCell")
example_graph.add_edge("A", "B")

graph_records = [
    {
        "patient_id": "1",
        "fov": "1",
        "graph": example_graph,
    }
]

# Uncomment to run.
# prep_result = prepare_from_graphs(
#     graphs=graph_records,
#     path_to_output_dir=str(OUTPUT_DATASET_DIR),
#     node_type_attribute="cell_type",
# )
# prep_result
# print_next_step_handoff(prep_result)


## Create `patient_class.csv`

After creating the graph `txt` files, make sure the same dataset folder also contains `patient_class.csv`.

Each row should look like:

```text
<dataset_prefix><patient_id>,<class_or_value>
```

Examples:

```text
CRC1,POSITIVE
CRC2,NEGATIVE
```

or:

```text
CRC1,2612
CRC2,3822
```

## Validate The Final Dataset Folder

Run this section after:

1. one of the three preparation routes
2. writing `patient_class.csv`

If validation succeeds, you are ready for the next notebook.

In [7]:
txt_files = validate_network_dataset_directory(OUTPUT_DATASET_DIR)
patient_class_path = OUTPUT_DATASET_DIR / "patient_class.csv"

print("Validated txt graph files:")
for file_name in txt_files[:10]:
    print(" -", file_name)
if len(txt_files) > 10:
    print(f" ... and {len(txt_files) - 10} more")

print()
print(f"patient_class.csv exists: {patient_class_path.exists()}")
print(f"Dataset folder ready: {OUTPUT_DATASET_DIR}")
print()
if "prep_result" in globals():
    print_next_step_handoff(prep_result)
else:
    print_next_step_handoff()


Validated txt graph files:
 - Patient_10_FOV10.txt
 - Patient_11_FOV11.txt
 - Patient_12_FOV12.txt
 - Patient_13_FOV13.txt
 - Patient_14_FOV14.txt
 - Patient_15_FOV15.txt
 - Patient_15_FOV22.txt
 - Patient_16_FOV16.txt
 - Patient_17_FOV17.txt
 - Patient_18_FOV18.txt
 ... and 28 more

patient_class.csv exists: True
Dataset folder ready: /home/omriavit/CISM/CISM_paper_revision/pet_data/TNBC

Use these values in tutorial 02:
network_dataset_root_path = /home/omriavit/CISM/CISM_paper_revision/pet_data
dataset_folder = TNBC
dataset directory on disk = /home/omriavit/CISM/CISM_paper_revision/pet_data/TNBC
number of graph txt files written = 38
preparation route = centroids
common_cells_type =
{0: 'B cells', 1: 'CD3 T cells', 2: 'CD4 t cells', 3: 'CD8 T cells', 4: 'DC', 5: 'DC/Mono', 6: 'Endothelial', 7: 'Immune other', 8: 'Macrophages', 9: 'Mesenchyme', 10: 'Mono/Neu', 11: 'NK cells', 12: 'Neutrophils', 13: 'Tregs', 14: 'Tumor', 15: 'Unidentified'}


## You Are Done When...

Before moving on, confirm all of the following:

- your dataset folder contains one `Patient_<patient_id>_FOV<fov>.txt` per FOV
- every txt row has exactly 4 values: `src_id dst_id src_type_id dst_type_id`
- the same folder contains `patient_class.csv`
- `validate_network_dataset_directory(...)` runs without errors

At that point, your output folder is ready for the next pipeline stage:

```python
cism.add_dataset(dataset_folder=..., dataset_type=..., dataset_name=...)
```

Recommended next notebook: **FANMOD+ and CISM initialization**.